In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import norm


In [2]:
def make_nf4_codebook():
    """
    Build the 16-value NF4 codebook: quantiles of a standard normal
    distribution, asymmetric (8 negative, exact 0, 7 positive) so that
    zero is exactly representable.
    """
    offset = 0.9677083
    neg_probs = torch.linspace(1 - offset, 0.5, 9)[:-1]
    neg = norm.ppf(neg_probs.tolist())
    pos_probs = torch.linspace(0.5, offset, 8)[1:]
    pos = norm.ppf(pos_probs.tolist())
    codebook = torch.tensor(list(neg) + [0.0] + list(pos), dtype=torch.float32)
    codebook = torch.sort(codebook).values
    codebook = codebook / codebook.abs().max()
    return codebook


In [3]:
def quantize_nf4(tensor, codebook, block_size=64):
    device = tensor.device
    codebook = codebook.to(device)
    flat = tensor.flatten()
    n = flat.numel()
    pad = (-n) % block_size
    if pad:
        flat = torch.cat([flat, torch.zeros(pad, device=device)])
    blocks = flat.view(-1, block_size)
    scales = blocks.abs().max(dim=1, keepdim=True).values.clamp(min=1e-8)
    normalized = blocks / scales
    diffs = (normalized.unsqueeze(-1) - codebook.view(1, 1, -1)).abs()
    indices = diffs.argmin(dim=-1).to(torch.uint8)
    return indices, scales.squeeze(1), tensor.shape, n

def dequantize_nf4(indices, scales, codebook, orig_shape, orig_numel, block_size=64):
    codebook = codebook.to(indices.device)
    looked_up = codebook[indices.long()]
    dequantized = looked_up * scales.unsqueeze(1)
    flat = dequantized.flatten()[:orig_numel]
    return flat.view(orig_shape)

def quantize_scales_8bit(scales, block_size=256):
    device = scales.device
    mean = scales.mean()
    centered = scales - mean
    n = centered.numel()
    pad = (-n) % block_size
    if pad:
        centered = torch.cat([centered, torch.zeros(pad, device=device)])
    blocks = centered.view(-1, block_size)
    block_scales = blocks.abs().max(dim=1, keepdim=True).values.clamp(min=1e-8)
    normalized = blocks / block_scales
    levels = torch.linspace(-1, 1, 256, device=device)
    diffs = (normalized.unsqueeze(-1) - levels.view(1, 1, -1)).abs()
    indices = diffs.argmin(dim=-1).to(torch.uint8)
    return indices, block_scales.squeeze(1), mean, n

def dequantize_scales_8bit(indices, block_scales, mean, orig_numel, block_size=256):
    levels = torch.linspace(-1, 1, 256, device=indices.device)
    looked_up = levels[indices.long()]
    dequantized = looked_up * block_scales.unsqueeze(1)
    flat = dequantized.flatten()[:orig_numel]
    return flat + mean

In [4]:
# def quantize_nf4(tensor: torch.Tensor, codebook: torch.Tensor, block_size: int = 64):
#     """
#     Quantize a float tensor to 4-bit NF4 indices, block-wise.
#     Returns: indices (uint8), scales (per-block absmax), orig_shape, orig_numel.
#     """
#     flat = tensor.flatten()
#     n = flat.numel()
#     pad = (-n) % block_size
#     if pad:
#         flat = torch.cat([flat, torch.zeros(pad)])
#     blocks = flat.view(-1, block_size)

#     scales = blocks.abs().max(dim=1, keepdim=True).values
#     scales = scales.clamp(min=1e-8)
#     normalized = blocks / scales

#     diffs = (normalized.unsqueeze(-1) - codebook.view(1, 1, -1)).abs()
#     indices = diffs.argmin(dim=-1).to(torch.uint8)

#     return indices, scales.squeeze(1), tensor.shape, n

In [5]:
# def dequantize_nf4(indices: torch.Tensor, scales: torch.Tensor, codebook: torch.Tensor, orig_shape, orig_numel: int, block_size: int = 64):
#     """Reverse of quantize_nf4: index lookup, then rescale by the block's absmax."""
#     looked_up = codebook[indices.long()]
#     dequantized = looked_up * scales.unsqueeze(1)
#     flat = dequantized.flatten()[:orig_numel]
#     return flat.view(orig_shape)

In [6]:
def naive_int4_quantize(tensor, block_size=64):
    flat = tensor.flatten()
    n = flat.numel()
    pad = (-n) % block_size
    if pad:
        flat = torch.cat([flat, torch.zeros(pad)])
    blocks = flat.view(-1, block_size)
    scales = blocks.abs().max(dim=1, keepdim=True).values.clamp(min=1e-8)
    normalized = blocks / scales

    levels = torch.linspace(-1, 1, 16)   # <-- the ONLY real difference from NF4
    diffs = (normalized.unsqueeze(-1) - levels.view(1, 1, -1)).abs()
    idx = diffs.argmin(dim=-1)

    dequant = levels[idx] * scales
    return dequant.flatten()[:n].view(tensor.shape)

In [7]:
if __name__ == "__main__":
    cb = make_nf4_codebook()
    print("Codebook:", cb)
 
    # --- small readable test (8x4, block_size=8) ---
    torch.manual_seed(0)
    test_tensor = torch.randn(4, 8) * 0.02
 
    indices, scales, shape, n = quantize_nf4(test_tensor, cb, block_size=8)
    reconstructed = dequantize_nf4(indices, scales, cb, shape, n, block_size=8)
 
    print("\n--- Small test tensor (4x8, block_size=8) ---")
    print("Original:\n", test_tensor)
    print("Indices:\n", indices)
    print("Scales:\n", scales)
    print("Reconstructed:\n", reconstructed)
    print("Max absolute error:", (test_tensor - reconstructed).abs().max().item())
 
    naive_reconstructed = naive_int4_quantize(test_tensor, block_size=8)
    nf4_mse_small = torch.mean((test_tensor - reconstructed) ** 2).item()
    int4_mse_small = torch.mean((test_tensor - naive_reconstructed) ** 2).item()
    print(f"\n[Small tensor, N=32 -- too few samples for the statistical NF4")
    print(f" advantage to show reliably, kept here only for illustration]")
    print(f"NF4 MSE: {nf4_mse_small:.10f}   Naive INT4 MSE: {int4_mse_small:.10f}")
 
    # --- large realistic test (1024x1024, block_size=64) ---
    torch.manual_seed(0)
    big_tensor = torch.randn(1024, 1024) * 0.02  # ~1M values, weight-matrix-like scale
 
    indices, scales, shape, n = quantize_nf4(big_tensor, cb, block_size=64)
    nf4_recon = dequantize_nf4(indices, scales, cb, shape, n, block_size=64)
    int4_recon = naive_int4_quantize(big_tensor, block_size=64)
 
    nf4_mse = torch.mean((big_tensor - nf4_recon) ** 2).item()
    int4_mse = torch.mean((big_tensor - int4_recon) ** 2).item()
 
    print("\n--- Large tensor (1024x1024 = 1,048,576 values, block_size=64) ---")
    print("Original (top-left 8x8 slice):\n", big_tensor[:8, :8])
    print("\nNF4 reconstructed (same slice):\n", nf4_recon[:8, :8])
    print("\nNaive INT4 reconstructed (same slice):\n", int4_recon[:8, :8])
 
    print(f"\n--- Full tensor stats (all {big_tensor.numel()} values) ---")
    print(f"NF4 MSE:        {nf4_mse:.10f}")
    print(f"Naive INT4 MSE: {int4_mse:.10f}")
    print(f"NF4 improvement: {(1 - nf4_mse/int4_mse)*100:.2f}%")
 
    # --- compression ratio check ---
    orig_bytes = big_tensor.numel() * 4  # fp32
    packed_bytes = (big_tensor.numel() * 4 // 8) + (scales.numel() * 4)  # 4-bit/val + fp32 scales
    print(f"\nOriginal (fp32): {orig_bytes/1e6:.2f} MB")
    print(f"NF4 packed (4-bit/val + scales): {packed_bytes/1e6:.2f} MB")
    print(f"Compression ratio: {orig_bytes/packed_bytes:.2f}x")
 

Codebook: tensor([-1.0000, -0.7230, -0.5626, -0.4407, -0.3379, -0.2461, -0.1609, -0.0796,
         0.0000,  0.0911,  0.1848,  0.2844,  0.3949,  0.5251,  0.6962,  1.0000])

--- Small test tensor (4x8, block_size=8) ---
Original:
 tensor([[-0.0225, -0.0230, -0.0050, -0.0087,  0.0170,  0.0138, -0.0063, -0.0423],
        [ 0.0064, -0.0253,  0.0070,  0.0062,  0.0024,  0.0248,  0.0223, -0.0049],
        [-0.0271, -0.0339,  0.0113,  0.0159,  0.0120, -0.0311, -0.0068,  0.0371],
        [ 0.0150, -0.0117, -0.0035,  0.0037,  0.0278,  0.0317,  0.0189, -0.0169]])
Indices:
 tensor([[ 2,  2,  7,  5, 12, 11,  6,  0],
        [11,  0, 11, 11,  9, 15, 15,  6],
        [ 1,  0, 11, 12, 11,  1,  6, 15],
        [13,  4,  7,  9, 15, 15, 13,  2]], dtype=torch.uint8)
Scales:
 tensor([0.0423, 0.0253, 0.0371, 0.0317])
Reconstructed:
 tensor([[-0.0238, -0.0238, -0.0034, -0.0104,  0.0167,  0.0120, -0.0068, -0.0423],
        [ 0.0072, -0.0253,  0.0072,  0.0072,  0.0023,  0.0253,  0.0253, -0.0041],
        [-0.02

In [8]:
# def quantize_scales_8bit(scales: torch.Tensor, block_size: int = 256):
#     """
#     Double quantization: compress the level-1 (per-64-block) NF4 scales
#     down to 8-bit, using plain affine (uniform) quantization -- not NF4,
#     # since scale values are positive-only and don't follow a zero-centered
#     # Gaussian shape.
#     # """
    # mean = scales.mean()
    # centered = scales - mean  # remove the positive-only bias

    # n = centered.numel()
    # pad = (-n) % block_size
    # if pad:
    #     centered = torch.cat([centered, torch.zeros(pad)])
    # blocks = centered.view(-1, block_size)

    # block_scales = blocks.abs().max(dim=1, keepdim=True).values.clamp(min=1e-8)
    # normalized = blocks / block_scales  # roughly [-1, 1]

    # # uniform 8-bit levels: 256 evenly spaced points from -1 to 1
    # levels = torch.linspace(-1, 1, 256)
    # diffs = (normalized.unsqueeze(-1) - levels.view(1, 1, -1)).abs()
    # indices = diffs.argmin(dim=-1).to(torch.uint8)

    # return indices, block_scales.squeeze(1), mean, n

In [9]:
# def dequantize_scales_8bit(indices: torch.Tensor, block_scales: torch.Tensor,
#                             mean: torch.Tensor, orig_numel: int, block_size: int = 256):
#     """Reverse of quantize_scales_8bit: reconstruct the original level-1 scales."""
#     levels = torch.linspace(-1, 1, 256)
#     looked_up = levels[indices.long()]
#     dequantized = looked_up * block_scales.unsqueeze(1)
#     flat = dequantized.flatten()[:orig_numel]
#     return flat + mean

In [10]:


class QuantizedLinear(nn.Module):
    def __init__(self, weight: torch.Tensor, bias: torch.Tensor, codebook: torch.Tensor,
                 block_size: int = 64, scale_block_size: int = 256):
        super().__init__()
        self.codebook = codebook
        self.block_size = block_size
        self.scale_block_size = scale_block_size
        self.out_features, self.in_features = weight.shape

        indices, scales, orig_shape, orig_numel = quantize_nf4(weight, codebook, block_size)
        scale_indices, scale_block_scales, scale_mean, scale_numel = quantize_scales_8bit(
            scales, scale_block_size
        )

        self.register_buffer("indices", indices)
        self.register_buffer("scale_indices", scale_indices)
        self.register_buffer("scale_block_scales", scale_block_scales)
        self.register_buffer("scale_mean", scale_mean)
        self.orig_shape = orig_shape
        self.orig_numel = orig_numel
        self.scale_numel = scale_numel

        if bias is not None:
            self.register_buffer("bias", bias)
        else:
            self.bias = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scales = dequantize_scales_8bit(
            self.scale_indices, self.scale_block_scales, self.scale_mean,
            self.scale_numel, self.scale_block_size
        )
        weight = dequantize_nf4(
            self.indices, scales, self.codebook,
            self.orig_shape, self.orig_numel, self.block_size
        )
        return F.linear(x, weight.to(x.dtype), self.bias)


    @classmethod
    def from_linear(cls, linear: nn.Linear, codebook: torch.Tensor,
                     block_size: int = 64, scale_block_size: int = 256):
        return cls(linear.weight.data, linear.bias.data if linear.bias is not None else None,
                    codebook, block_size, scale_block_size)

In [11]:
import gc

def quantize_model_inplace(model, codebook, target_modules=["q_proj", "v_proj"]):
    num_layers = len(model.model.layers)
    
    for i, layer in enumerate(model.model.layers):
        attn = layer.self_attn
        
        for name in target_modules:
            orig = getattr(attn, name)
            setattr(attn, name, QuantizedLinear.from_linear(orig, codebook))
        
        gc.collect()
        torch.cuda.empty_cache()
        print(f"[{i+1}/{num_layers}] done")
    
    print("quantization complete.")


In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [13]:
cb = make_nf4_codebook()
quantize_model_inplace(model, cb, target_modules=["q_proj", "v_proj"])


[1/28] done
[2/28] done
[3/28] done
[4/28] done
[5/28] done
[6/28] done
[7/28] done
[8/28] done
[9/28] done
[10/28] done
[11/28] done
[12/28] done
[13/28] done
[14/28] done
[15/28] done
[16/28] done
[17/28] done
[18/28] done
[19/28] done
[20/28] done
[21/28] done
[22/28] done
[23/28] done
[24/28] done
[25/28] done
[26/28] done
[27/28] done
[28/28] done
quantization complete.


In [14]:
# Test: can the quantized model do a forward pass without crashing?
model.eval()

prompt = "SELECT * FROM"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

print("Output logits shape:", outputs.logits.shape)
print("No NaN in logits:", not torch.isnan(outputs.logits).any().item())
print("Forward pass successful!")


Output logits shape: torch.Size([1, 3, 152064])
No NaN in logits: True
Forward pass successful!


In [15]:
import math

class LoRALinear(nn.Module):
    def __init__(self, quantized_linear: QuantizedLinear, r: int = 8, alpha: int = 16):
        super().__init__()

        self.base = quantized_linear
        self.r = r
        self.scaling = alpha / r

        in_features = quantized_linear.in_features
        out_features = quantized_linear.out_features

        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))

        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        for param in self.base.parameters():
            param.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base(x)
        lora_out = F.linear(
            F.linear(x, self.lora_A.to(device=x.device, dtype=x.dtype)),
            self.lora_B.to(device=x.device, dtype=x.dtype)
        ) * self.scaling
        return base_out + lora_out



In [16]:
def inject_lora(model, r=8, alpha=16, target_modules=["q_proj", "v_proj"]):
    num_layers = len(model.model.layers)

    for i, layer in enumerate(model.model.layers):
        attn = layer.self_attn

        for name in target_modules:
            orig = getattr(attn, name)
            setattr(attn, name, LoRALinear(orig, r=r, alpha=alpha))

        print(f"[{i+1}/{num_layers}] LoRA injected")

    print("LoRA injection complete.")


In [17]:
inject_lora(model, r=8, alpha=16, target_modules=["q_proj", "v_proj"])


[1/28] LoRA injected
[2/28] LoRA injected
[3/28] LoRA injected
[4/28] LoRA injected
[5/28] LoRA injected
[6/28] LoRA injected
[7/28] LoRA injected
[8/28] LoRA injected
[9/28] LoRA injected
[10/28] LoRA injected
[11/28] LoRA injected
[12/28] LoRA injected
[13/28] LoRA injected
[14/28] LoRA injected
[15/28] LoRA injected
[16/28] LoRA injected
[17/28] LoRA injected
[18/28] LoRA injected
[19/28] LoRA injected
[20/28] LoRA injected
[21/28] LoRA injected
[22/28] LoRA injected
[23/28] LoRA injected
[24/28] LoRA injected
[25/28] LoRA injected
[26/28] LoRA injected
[27/28] LoRA injected
[28/28] LoRA injected
LoRA injection complete.


In [18]:
# Step 1: Freeze every single parameter in the model
for param in model.parameters():
    param.requires_grad = False

# Step 2: Unfreeze only the LoRA adapter parameters
for name, param in model.named_parameters():
    if "lora_A" in name or "lora_B" in name:
        param.requires_grad = True

# Step 3: Verify
trainable = [(n, p.shape, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
frozen    = [n for n, p in model.named_parameters() if not p.requires_grad]

total_trainable = sum(p[2] for p in trainable)
print(f"Trainable: {len(trainable)} params, {total_trainable:,} values")
print(f"Frozen:    {len(frozen)} params")
print()
for name, shape, count in trainable[:6]:
    print(f"  {name}  shape={shape}")


Trainable: 112 params, 2,523,136 values
Frozen:    227 params

  model.layers.0.self_attn.q_proj.lora_A  shape=torch.Size([8, 3584])
  model.layers.0.self_attn.q_proj.lora_B  shape=torch.Size([3584, 8])
  model.layers.0.self_attn.v_proj.lora_A  shape=torch.Size([8, 3584])
  model.layers.0.self_attn.v_proj.lora_B  shape=torch.Size([512, 8])
  model.layers.1.self_attn.q_proj.lora_A  shape=torch.Size([8, 3584])
  model.layers.1.self_attn.q_proj.lora_B  shape=torch.Size([3584, 8])


In [19]:
model.eval()

prompt = "SELECT * FROM"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

print("Output logits shape:", outputs.logits.shape)
print("No NaN in logits:", not torch.isnan(outputs.logits).any().item())
print("Forward pass successful!")


Output logits shape: torch.Size([1, 3, 152064])
No NaN in logits: True
Forward pass successful!


In [20]:
trainable = []
frozen = []

for name, param in model.named_parameters():
    if param.requires_grad:
        trainable.append((name, param.shape, param.numel()))
    else:
        frozen.append(name)

print(f"Trainable parameters: {len(trainable)}")
print(f"Frozen parameters:    {len(frozen)}")
print()

total_trainable = sum(p[2] for p in trainable)
print(f"Total trainable param count: {total_trainable:,}")
print()

print("Trainable parameter names:")
for name, shape, count in trainable[:10]:
    print(f"  {name}  shape={shape}  count={count:,}")


Trainable parameters: 112
Frozen parameters:    227

Total trainable param count: 2,523,136

Trainable parameter names:
  model.layers.0.self_attn.q_proj.lora_A  shape=torch.Size([8, 3584])  count=28,672
  model.layers.0.self_attn.q_proj.lora_B  shape=torch.Size([3584, 8])  count=28,672
  model.layers.0.self_attn.v_proj.lora_A  shape=torch.Size([8, 3584])  count=28,672
  model.layers.0.self_attn.v_proj.lora_B  shape=torch.Size([512, 8])  count=4,096
  model.layers.1.self_attn.q_proj.lora_A  shape=torch.Size([8, 3584])  count=28,672
  model.layers.1.self_attn.q_proj.lora_B  shape=torch.Size([3584, 8])  count=28,672
  model.layers.1.self_attn.v_proj.lora_A  shape=torch.Size([8, 3584])  count=28,672
  model.layers.1.self_attn.v_proj.lora_B  shape=torch.Size([512, 8])  count=4,096
  model.layers.2.self_attn.q_proj.lora_A  shape=torch.Size([8, 3584])  count=28,672
  model.layers.2.self_attn.q_proj.lora_B  shape=torch.Size([3584, 8])  count=28,672


In [21]:
from datasets import load_dataset

dataset = load_dataset("spider", trust_remote_code=True)
train_data = dataset["train"]

print(f"Training examples: {len(train_data)}")
print("\nSample example:")
print("Question :", train_data[0]["question"])
print("SQL      :", train_data[0]["query"])
print("DB       :", train_data[0]["db_id"])


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'spider' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

Training examples: 7000

Sample example:
Question : How many heads of the departments are older than 56 ?
SQL      : SELECT count(*) FROM head WHERE age  >  56
DB       : department_management


In [22]:
def tokenize_example(example, tokenizer, max_length=256):
    # Build the two parts separately
    prompt = (
        f"### Task: Convert the question to SQL.\n"
        f"### Database: {example['db_id']}\n"
        f"### Question: {example['question']}\n"
        f"### SQL:\n"
    )
    answer = example["query"] + tokenizer.eos_token

    # Tokenize each part separately so we know where the boundary is
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]

    input_ids = prompt_ids + answer_ids

    # Labels: -100 for the prompt (ignore), real token ids for the SQL answer
    labels = [-100] * len(prompt_ids) + answer_ids

    # Pad or truncate to max_length
    input_ids = input_ids[:max_length]
    labels    = labels[:max_length]

    pad_len = max_length - len(input_ids)
    input_ids = input_ids + [tokenizer.pad_token_id] * pad_len
    labels    = labels    + [-100] * pad_len

    return {
        "input_ids": torch.tensor(input_ids),
        "labels":    torch.tensor(labels),
    }


In [23]:
tokenized = [tokenize_example(ex, tokenizer) for ex in train_data]
print(f"Tokenized {len(tokenized)} examples")
print(f"Input shape: {tokenized[0]['input_ids'].shape}")


Tokenized 7000 examples
Input shape: torch.Size([256])


In [24]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "labels":    torch.stack([b["labels"]    for b in batch]),
    }

train_loader = DataLoader(
    tokenized,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

print(f"Total batches per epoch: {len(train_loader)}")


Total batches per epoch: 3500


In [25]:
val_data = dataset["validation"]
val_tokenized = [tokenize_example(ex, tokenizer) for ex in val_data]

val_loader = DataLoader(
    val_tokenized,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)

print(f"Val examples: {len(val_tokenized)}")
print(f"Val batches:  {len(val_loader)}")


Val examples: 1034
Val batches:  517


In [26]:
# from torch.optim import AdamW
# from tqdm import tqdm



# optimizer = AdamW(
#     [p for p in model.parameters() if p.requires_grad],
#     lr=2e-4
# )

# num_epochs = 1
# log_every  = 100  # every 100 steps: show both train loss AND val loss

# def run_validation(model, val_loader):
#     model.eval()
#     total_val_loss = 0.0
#     with torch.no_grad():
#         for batch in val_loader:
#             input_ids = batch["input_ids"].to(model.device)
#             labels    = batch["labels"].to(model.device)
#             total_val_loss += model(input_ids=input_ids, labels=labels).loss.item()
#     model.train()
#     return total_val_loss / len(val_loader)


# model.train()

# for epoch in range(num_epochs):
#     total_train_loss = 0.0

#     for step, batch in enumerate(tqdm(train_loader)):
#         input_ids = batch["input_ids"].to(model.device)
#         labels    = batch["labels"].to(model.device)

#         optimizer.zero_grad()
#         outputs = model(input_ids=input_ids, labels=labels)
#         loss = outputs.loss
#         loss.backward()
#         optimizer.step()

#         total_train_loss += loss.item()

#         if (step + 1) % log_every == 0:
#             avg_train = total_train_loss / (step + 1)
#             val_loss  = run_validation(model, val_loader)
#             print(
#                 f"Epoch {epoch+1} | Step {step+1}/{len(train_loader)} | "
#                 f"train_loss: {avg_train:.4f} | val_loss: {val_loss:.4f}",
#                 flush=True
#             )

#     # End of epoch summary
#     val_loss  = run_validation(model, val_loader)
#     avg_train = total_train_loss / len(train_loader)
#     print(f"\nEpoch {epoch+1} complete.", flush=True)
#     print(f"  avg train loss : {avg_train:.4f}", flush=True)
#     print(f"  final val loss : {val_loss:.4f}", flush=True)


In [31]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm.notebook import tqdm

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4
)

num_training_steps = 1000
num_warmup_steps   = int(0.1 * num_training_steps)  # 100 warmup steps
val_steps          = 50
log_every          = 25

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

def run_validation(model, val_loader, max_steps=50):
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            if i >= max_steps:
                break
            input_ids = batch["input_ids"].to(model.device)
            labels    = batch["labels"].to(model.device)
            total_val_loss += model(input_ids=input_ids, labels=labels).loss.item()
    model.train()
    return total_val_loss / max_steps

best_val_loss    = float("inf")
total_train_loss = 0.0

model.train()

for step, batch in enumerate(tqdm(train_loader, desc="Training", total=num_training_steps)):
    if step >= num_training_steps:
        break

    input_ids = batch["input_ids"].to(model.device)
    labels    = batch["labels"].to(model.device)

    optimizer.zero_grad()
    loss = model(input_ids=input_ids, labels=labels).loss
    loss.backward()
    optimizer.step()
    scheduler.step()

    total_train_loss += loss.item()

    if (step + 1) % log_every == 0:
        avg_train = total_train_loss / (step + 1)
        val_loss  = run_validation(model, val_loader, max_steps=val_steps)
        print(
            f"\nStep {step+1}/{num_training_steps} | "
            f"train: {avg_train:.4f} | val: {val_loss:.4f}",
            flush=True
        )
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                "lora_state_dict": {k: v for k, v in model.state_dict().items() if "lora" in k},
                "step": step + 1,
                "val_loss": val_loss,
            }, "/kaggle/working/best_qlora_checkpoint.pt")
            print(f"  -> new best saved (val: {val_loss:.4f})", flush=True)

print("\nTraining complete.", flush=True)
print(f"Best val loss: {best_val_loss:.4f}")


Training:   0%|          | 0/1000 [00:00<?, ?it/s]


Step 25/1000 | train: 1.4055 | val: 1.2947
  -> new best saved (val: 1.2947)

Step 50/1000 | train: 1.3033 | val: 1.0769
  -> new best saved (val: 1.0769)

Step 75/1000 | train: 1.2227 | val: 0.7369
  -> new best saved (val: 0.7369)

Step 100/1000 | train: 1.0651 | val: 0.5287
  -> new best saved (val: 0.5287)

Step 125/1000 | train: 0.9697 | val: 0.5289

Step 150/1000 | train: 0.9154 | val: 0.4839
  -> new best saved (val: 0.4839)

Step 175/1000 | train: 0.8783 | val: 0.4742
  -> new best saved (val: 0.4742)

Step 200/1000 | train: 0.8342 | val: 0.4842

Step 225/1000 | train: 0.7980 | val: 0.4788

Step 250/1000 | train: 0.7691 | val: 0.4683
  -> new best saved (val: 0.4683)

Step 275/1000 | train: 0.7543 | val: 0.4680
  -> new best saved (val: 0.4680)

Step 300/1000 | train: 0.7348 | val: 0.4933

Step 325/1000 | train: 0.7198 | val: 0.4646
  -> new best saved (val: 0.4646)

Step 350/1000 | train: 0.7045 | val: 0.4676

Step 375/1000 | train: 0.6893 | val: 0.4713

Step 400/1000 | train